In [ ]:
import pandas as pd

df = pd.read_json('DATA_AUGMENTED_ENRICHED.jsonl', lines=True)

df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.info()

In [ ]:
missing_data = df.isnull().sum()

print(missing_data)

In [ ]:
df['task_items_versions'].apply(lambda x: [item['version_type'] for item in x]).head()

versions_count = df['task_items_versions'].apply(lambda x: [item['version_type'] for item in x]).explode().value_counts()
print(versions_count)

In [ ]:

df[['global_task_description', 'task_items_versions']].head(1)

In [ ]:

df['global_task_description_length'] = df['global_task_description'].apply(len)

df['task_items_length'] = df['task_items_versions'].apply(lambda x: sum([len(item['task_items']) for item in x]))

df[['global_task_description_length', 'task_items_length']].describe()

In [ ]:

def check_consistent_task_item_count(task_items_versions):

    version_lengths = {}
    for item in task_items_versions:
        version_lengths[item['version_type']] = len(item['task_items'])

    return len(set(version_lengths.values())) == 1, version_lengths

df['is_task_item_count_consistent'], df['task_item_lengths'] = zip(*df['task_items_versions'].apply(check_consistent_task_item_count))

df[['id', 'is_task_item_count_consistent', 'task_item_lengths']].head()

In [ ]:

inconsistent_rows = df[df['is_task_item_count_consistent'] == False]

inconsistent_rows[['id', 'is_task_item_count_consistent', 'task_item_lengths']]

In [ ]:
inconsistent_count = len(inconsistent_rows)
inconsistent_count

# Preprocess

## Décomposer la colonne task_items_versions

In [ ]:
import pandas as pd

def extract_task_versions(row):
    versions = {}

    for item in row:
        version_type = item['version_type']
        versions[version_type] = item['task_items']
    return versions

df_task_versions = df['task_items_versions'].apply(extract_task_versions)

df_task_versions_normalized = pd.json_normalize(df_task_versions)

df = df.join(df_task_versions_normalized)

In [ ]:
print(df[['id', 'original', 'paraphrase_1', 'paraphrase_2', 'tool_replaced', 'keyword_removed', 'global_task_description']].head())

## Nettoyage des tâches

In [ ]:
df.isnull().sum()

In [ ]:
import re

def normalize_syntax(text: str) -> str:
    """
    Nettoyage syntaxique léger sans perte de sens.
    """
    if not isinstance(text, str):
        return text

    text = text.replace('\\"', '"')
    text = text.replace('"', '')

    text = re.sub(r'[\u200b\u200e\u200f]', '', text)

    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text

def normalize_task_items(task_items):
    return [normalize_syntax(item) for item in task_items]

def normalize_paths(text):

    return re.sub(r"(/\S+)+", "<PATH>", text)

def semantic_normalize_item_correct(text):
    text_norm = text

    text_norm = normalize_paths(text_norm)
    return text_norm

def semantic_normalize_task_items_correct(task_items):
    return [semantic_normalize_item_correct(item) for item in task_items]

df['task_items_clean'] = df['original'].apply(normalize_task_items)

df["task_items_semantic"] = df["task_items_clean"].apply(semantic_normalize_task_items_correct)

print(df[['original', 'task_items_clean', 'task_items_semantic']].head())

In [ ]:
print(df.columns)

In [ ]:

variants = ['original', 'paraphrase_1', 'paraphrase_2', 'tool_replaced', 'keyword_removed']

data_rows = []

for _, row in df.iterrows():
    target = row['global_task_description']
    task_id = row['id']  # Pour identifier chaque ligne dans l'entraînement

    for var in variants:
        tasks_list = row[var]

        if isinstance(tasks_list, list):
            tasks_string = ", ".join(tasks_list)
        else:
            tasks_string = str(tasks_list)

        input_text = f"generate global description: {tasks_string}"

        data_rows.append({
            'task_id': task_id,  # Ajouter un identifiant de tâche pour traçabilité
            'input_text': input_text,
            'target_text': target
        })

df_data = pd.DataFrame(data_rows)

df_data = df_data.sample(frac=1).reset_index(drop=True)

## vérifier et gérer la longueur des tokens

In [ ]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-small")

def check_token_length(df, max_length=512):

    df['input_token_length'] = df['input_text'].apply(lambda x: len(tokenizer.encode(x, truncation=True, padding='max_length', max_length=max_length)))

    long_sequences = df[df['input_token_length'] > max_length]

    if len(long_sequences) > 0:
        print(f"Il y a {len(long_sequences)} exemples dont la longueur de tokens dépasse {max_length}.")
        print(long_sequences[['input_text', 'input_token_length']].head())
    else:
        print(f"Aucun exemple ne dépasse la longueur maximale de {max_length} tokens.")

    return df

df_data = check_token_length(df_data, max_length=512)

print(df_data[['input_text', 'input_token_length']].head())

In [ ]:
def build_prompt(task_items):
    joined = "\n".join([f"- {item}" for item in task_items])
    return f"""You are given a list of task items:

{joined}

Your goal is to infer the underlying global objective.

Step 1: Extract the essential key concepts (avoid surface words).
Step 2: Determine the broader domain these tasks belong to.
Step 3: Identify the single primary action that unifies all items.
Step 4: Generate one concise global task description that reflects a coherent higher-level objective.

Strict Output Format:

Key Concepts: [...]
Domain: ...
Main Action: ...
Global Task: ...
"""

df["input_text"] = df["task_items_semantic"].apply(build_prompt)
df["target_text"] = df["global_task_description"]

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["input_text", "target_text"]])

# Split en train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

## Preparation des donnees PyTorch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq


## Tokenisation améliorée

In [ ]:
from transformers import AutoTokenizer

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):

    model_inputs = tokenizer(
        example["input_text"],
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True
    )

    labels = tokenizer(
        example["target_text"],
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True
    )

    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

lora_config = LoraConfig(
    r=32,  # Rank
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM  # FLAN-T5
)


In [ ]:
from transformers import AutoModelForSeq2SeqLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float16)

peft_model = get_peft_model(original_model, lora_config)

output_dir = "/content/drive/MyDrive/flan_t5_LoRA"

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=peft_model
)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=50,
    logging_steps=50,
    eval_steps=50,
    save_total_limit=2,
    eval_strategy="steps",
)

peft_trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)


In [ ]:
peft_trainer.train()

peft_model_path = "/content/drive/MyDrive/augmented-peft-global_task-checkpoint-local"
peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel


device = "cuda" if torch.cuda.is_available() else "cpu"

base_model_name = "google/flan-t5-small"
adapter_path = "/content/drive/MyDrive/augmented-peft-global_task-checkpoint-local"


tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

model = PeftModel.from_pretrained(base_model, adapter_path)

model = model.to(device)
model.eval()

print(" Model loaded successfully")

In [ ]:
def generate_prediction(task_items, max_new_tokens=64):

    prompt = f"""
Generate a concise global task description from the following task items:

{task_items}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
            length_penalty=1.0,
            early_stopping=True
        )

    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return prediction.strip()

In [ ]:

all_task_items = [

"""
- Install Nmap
- Scan target IP
- Analyze open ports
- Identify running services
""",

"""
- Create index.html file
- Define page structure using HTML
- Style layout with CSS
- Add interactivity using JavaScript
""",

"""
- Initialize git repository
- Create new feature branch
- Commit incremental changes
- Merge branch into main
""",

"""
- Set up Node.js server
- Configure API routes
- Connect to MongoDB database
- Test endpoints using Postman
""",

"""
- Design UI mockups in Figma
- Implement responsive layout with Bootstrap
- Test UI across different screen sizes
- Debug CSS issues using browser developer tools
""",

"""
- Write unit tests for authentication module
- Run automated test suite
- Fix failing test cases
- Refactor code for better maintainability
""",

"""
- Create Dockerfile for application
- Build Docker image
- Run container locally
- Push image to Docker Hub
""",

"""
- Configure Nginx reverse proxy
- Enable HTTPS with SSL certificate
- Restart web server
- Verify secure connection in browser
""",

"""
- Define database schema
- Create tables and relationships
- Write SQL queries
- Optimize database performance
""",

"""
- Create ablation study script
- Remove selected model features
- Evaluate model performance after feature removal
- Compute and compare feature importance
- Analyze feature contributions using SHAP
- Record ablation results
""",

"""
- Install Metasploit framework
- Configure exploit module
- Set target host parameters
- Launch exploit against target system
- Establish reverse shell connection
""",

"""
- Capture network traffic using Wireshark
- Filter packets by suspicious IP address
- Analyze TCP handshake anomalies
- Inspect payload for malicious patterns
- Generate incident report
""",

"""
- Perform SQL injection testing on login form
- Intercept HTTP requests using Burp Suite
- Modify input parameters
- Analyze server responses for vulnerabilities
- Document discovered security flaws
""",

"""
- Configure firewall rules
- Block unauthorized incoming connections
- Monitor suspicious outbound traffic
- Review system security logs
- Update intrusion prevention policies
""",

"""
- Conduct password brute-force attack simulation
- Use Hydra to test credential combinations
- Analyze authentication failure logs
- Identify weak password policies
- Recommend security hardening measures
""",

"""
- Open Google Chrome
- Search for best restaurants near me
- Bookmark favorite restaurants
- Take screenshots of menus
- Close browser
""",

"""
- Open Microsoft Word
- Create a new document
- Type personal notes
- Save the document in Documents folder
- Close Microsoft Word
"""

]


In [ ]:
all_task_items = [

# 1 — Configure a home WiFi network
[
"router_admin.html in /config opened with Chrome to configure WiFi SSID and password",
"network_settings.conf in /router opened with Notepad to adjust DHCP range",
"ISP_dashboard website used to activate the internet connection",
"speedtest.net website used to measure internet bandwidth",
"router firmware update page used to install the latest firmware",
"WiFi Analyzer mobile app used to detect channel interference",
"connect laptop to WiFi network using system network settings",
"ping 8.8.8.8 command used to test internet connectivity",
"restart router device to apply network configuration changes"
],

# 2 — Train a machine learning model
[
"train_model.py in /ml project opened with VS Code to implement neural network training",
"dataset.csv in /data opened with Jupyter Notebook to explore features and labels",
"requirements.txt file used to install PyTorch and pandas dependencies",
"TensorBoard dashboard used to visualize training metrics",
"python train_model.py command used to launch model training",
"scaler.pkl file generated to normalize input features",
"model_weights.pt file saved after training completion",
"Kaggle website used to download the dataset",
"git commit command used to version the trained model code"
],

# 3 — Perform web application penetration testing
[
"nmap command used to scan open ports on the target server",
"burpsuite application used to intercept HTTP requests",
"sqlmap tool executed to test SQL injection vulnerabilities",
"dirsearch script used to enumerate hidden directories",
"nikto scanner used to detect outdated web server software",
"hydra tool used to attempt brute force login testing",
"OWASP ZAP used to analyze web application security issues",
"curl command used to manually test API endpoints",
"report.md file in /pentest opened with VS Code to document vulnerabilities"
],

# 4 — Build a REST API
[
"server.js file in /api opened with Visual Studio Code to implement REST endpoints",
"routes.js file in /api/routes opened to define API routes",
"package.json file used to install Express dependencies",
"Postman application used to test API endpoints",
"MongoDB Compass used to inspect database collections",
"npm install express command executed to install server framework",
"node server.js command used to start the API server",
"docker-compose.yml file used to configure API container deployment",
"git push origin main used to deploy API code to repository"
],

# 5 — Edit and publish a YouTube video
[
"raw_footage.mp4 file opened in Adobe Premiere Pro to start video editing",
"audio_track.wav imported into the editing timeline",
"color_correction panel used to improve video lighting",
"export_settings window used to render the final video",
"YouTube Studio website used to upload the edited video",
"thumbnail.png created in Photoshop for the video cover",
"description.txt file used to write video metadata",
"upload progress page used to publish the video",
"analytics dashboard used to monitor video views"
],

# 6 — Play and stream a video game
[
"OBS Studio opened to configure game streaming scene",
"Twitch dashboard used to start a live stream",
"Discord application used for voice chat with teammates",
"game_settings menu used to adjust graphics quality",
"keyboard macro software used to configure gaming shortcuts",
"Steam client used to launch the game",
"FPS counter overlay used to monitor game performance",
"chat window used to interact with viewers during the stream",
"stream title edited on Twitch before going live"
],

# 7 — Manage a Linux server
[
"ssh user@server command used to access the remote server",
"nginx.conf file in /etc/nginx opened with nano to configure web server",
"systemctl restart nginx command used to apply configuration",
"top command used to monitor server processes",
"apt update command executed to refresh package lists",
"ufw enable command used to activate firewall protection",
"/var/log/syslog file opened to inspect system logs",
"crontab -e command used to schedule maintenance tasks",
"htop tool used to visualize system resource usage"
],

# 8 — Design a mobile application interface
[
"mobile_wireframe.fig file opened in Figma to design UI layout",
"color_palette.png used to define application theme colors",
"component_library.fig used to reuse interface components",
"prototype mode used to simulate user navigation",
"export assets option used to generate PNG icons",
"Slack workspace used to share design updates with developers",
"design_review.md document used to record UI feedback",
"Adobe Illustrator used to refine vector icons",
"handoff panel used to provide design specs for developers"
],

# 9 — Analyze business sales data
[
"sales_data.xlsx file opened in Excel to analyze revenue metrics",
"pivot_table feature used to summarize monthly sales",
"powerbi dashboard used to visualize trends",
"python analysis_script.py executed to compute KPIs",
"matplotlib library used to generate charts",
"SQL query executed to retrieve sales records from database",
"report.docx document used to summarize insights",
"Google Sheets used to share results with stakeholders",
"forecast_model.ipynb notebook used to predict future sales"
],

# 10 — Create a cybersecurity monitoring dashboard
[
"logstash.conf file opened with VS Code to configure log ingestion",
"kibana dashboard used to visualize security events",
"filebeat.yml file configured to ship system logs",
"SIEM platform used to detect suspicious activity",
"alert_rules.json created to define security alerts",
"ElasticSearch index used to store security logs",
"dashboard.json exported for monitoring configuration",
"Python script used to automate log parsing",
"Slack webhook used to send security alerts"
],

# 11 — Install and configure a database
[
"postgresql.conf file in /etc/postgresql opened with nano",
"pg_hba.conf file edited to configure client authentication",
"psql terminal used to create new database",
"CREATE TABLE SQL command used to define schema",
"backup.sql file used to restore database backup",
"pgAdmin interface used to manage database objects",
"systemctl start postgresql command used to start service",
"database logs opened to inspect connection errors",
"cron job created to automate database backups"
],

# 12 — Shop online for products
[
"Amazon website opened to browse electronic products",
"search bar used to find wireless headphones",
"product page opened to compare specifications",
"reviews section read to evaluate product quality",
"shopping cart page used to review selected items",
"payment form filled with credit card information",
"order confirmation email received after purchase",
"delivery tracking page used to follow shipment",
"wishlist feature used to save items for later"
],

# 13 — Create a personal blog website
[
"index.html file opened in VS Code to build homepage layout",
"style.css file edited to customize blog design",
"blog_post.md file created to write a new article",
"GitHub Pages used to host the blog website",
"jekyll serve command used to preview the blog locally",
"favicon.png file added to website assets",
"SEO plugin configured to optimize search ranking",
"domain settings updated to link custom domain",
"Google Analytics installed to track blog visitors"
],

# 14 — Develop a mobile game
[
"game_scene.unity file opened in Unity Editor to design level",
"player_controller.cs script written to control character movement",
"sprite_sheet.png imported for game animations",
"physics settings configured for collision detection",
"sound_effect.wav added to game audio system",
"build settings used to export Android APK",
"debug console used to test gameplay mechanics",
"Git repository used to version the game project",
"Play Store dashboard used to publish the game"
],

# 15 — Automate cloud infrastructure deployment
[
"terraform main.tf file opened with VS Code to define infrastructure",
"AWS console used to inspect cloud resources",
"terraform init command executed to initialize project",
"terraform apply used to deploy infrastructure",
"variables.tf file created to parameterize configuration",
"cloudwatch dashboard used to monitor services",
"docker image built for application deployment",
"CI/CD pipeline configured in GitHub Actions",
"infrastructure_state.tfstate file used to track deployments"
]

]

In [ ]:


for i, task_items in enumerate(all_task_items):
    prediction = generate_prediction(task_items)

    print("="*80)
    print(f"Example {i}")
    print("Task Items:")
    print(task_items.strip())
    print("\nPredicted Global Task Description:")
    print("PREDICTION :", prediction)
    print("="*80)

In [ ]:
for i, task_items in enumerate(all_task_items):

    task_items_text = "\n".join(task_items)

    prediction = generate_prediction(task_items_text)

    print("="*80)
    print(f"Example {i}")
    print("Task Items:")
    print(task_items_text)

    print("\nPredicted Global Task Description:")
    print("PREDICTION :", prediction)
    print("="*80)